In [ ]:
# ============================================================
# MEDIVOICE - Gemma 4 Good Hackathon | github.com/hamnamgl/Medivoice
# Offline AI Health Copilot for Frontline Community Health Workers
# Kaggle notebook: reproducible demo mode aligned with repo main
# ============================================================

import json
import os
import subprocess
import sys
import threading
import time
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from pathlib import Path

import requests
from IPython.display import HTML, display

print("NOTE: This Kaggle notebook is a reproducible demo environment.")
print("NOTE: True offline deployment runs the same MediVoice stack locally with Ollama on-device.")
print("NOTE: Demo mode may use ngrok for browser access, but the consultation stack itself stays repo-aligned.")

REPO_URL = "https://github.com/hamnamgl/Medivoice.git"
REPO = Path("/kaggle/working/Medivoice")
OLLAMA_BIN = "/usr/local/bin/ollama"
OLLAMA_HOST = "http://localhost:11434"
PROXY_PORT = 8000
PRIMARY_MODEL = "gemma3:4b"
FALLBACK_MODEL = "gemma4:e4b"


def run_shell(command, label, check=False):
    result = subprocess.run(command, shell=True, text=True, capture_output=True)
    if result.returncode == 0:
        print(f"[ok] {label}")
    else:
        print(f"[warn] {label} failed (exit {result.returncode})")
        if result.stderr.strip():
            print(result.stderr.strip()[-500:])
        if check:
            raise RuntimeError(f"Command failed: {command}")
    return result


print("\n== Repo sync ==")
if not REPO.exists():
    run_shell(f"git clone {REPO_URL} {REPO}", "Repo cloned", check=True)
else:
    run_shell(f"git -C {REPO} pull", "Repo updated")

sys.path.insert(0, str(REPO))
os.chdir(REPO)

from app.utils.config import APP_MODE, DEFAULT_MODEL, FALLBACK_MODEL, SUPPORTED_LANGUAGES

print(f"[ok] Repo mode: {APP_MODE}")
print(f"[ok] Repo default model: {DEFAULT_MODEL} | fallback: {FALLBACK_MODEL}")
print(f"[ok] Supported UI languages in repo: {len(SUPPORTED_LANGUAGES)}")

print("\n== Python dependencies ==")
run_shell(
    "pip install -q ollama openai-whisper edge-tts requests pyngrok streamlit > /tmp/medivoice_pip.log 2>&1",
    "Dependencies installed"
)

print("\n== System dependency ==")
run_shell("apt-get install -y -q zstd > /tmp/medivoice_apt.log 2>&1", "zstd installed")

print("\n== Ollama setup ==")
if not os.path.exists(OLLAMA_BIN):
    print("Installing Ollama...")
    install_result = run_shell(
        "curl -fsSL https://ollama.com/install.sh | sh > /tmp/medivoice_ollama_install.log 2>&1",
        "Ollama installed"
    )
    if install_result.returncode != 0 or not os.path.exists(OLLAMA_BIN):
        raise RuntimeError("Ollama install failed. Check /tmp/medivoice_ollama_install.log")
else:
    print("[ok] Ollama already installed - skipping")


def start_ollama():
    os.system("ollama serve > /tmp/ollama.log 2>&1")


threading.Thread(target=start_ollama, daemon=True).start()
print("Waiting for Ollama server to start...")
for _ in range(40):
    try:
        response = requests.get(f"{OLLAMA_HOST}/", timeout=2)
        if response.status_code == 200:
            print("[ok] Ollama server is running")
            break
    except Exception:
        pass
    time.sleep(1)
else:
    raise RuntimeError("Ollama server did not start in time. Check /tmp/ollama.log")


def model_is_cached(model_name):
    try:
        response = requests.get(f"{OLLAMA_HOST}/api/tags", timeout=5)
        models = response.json().get("models", [])
        return any(item.get("name", "") == model_name for item in models)
    except Exception:
        return False


selected_model = PRIMARY_MODEL
if model_is_cached(PRIMARY_MODEL):
    print(f"[ok] {PRIMARY_MODEL} already cached - skipping download")
elif model_is_cached(FALLBACK_MODEL):
    selected_model = FALLBACK_MODEL
    print(f"[ok] {FALLBACK_MODEL} already cached - using fallback model for this demo")
else:
    print(f"Downloading {PRIMARY_MODEL} - only once, cached after this...")
    pull_result = run_shell(
        f"ollama pull {PRIMARY_MODEL} > /tmp/medivoice_ollama_pull.log 2>&1",
        f"{PRIMARY_MODEL} downloaded"
    )
    if pull_result.returncode != 0:
        print("[warn] Primary model download failed. Trying fallback model...")
        fallback_result = run_shell(
            f"ollama pull {FALLBACK_MODEL} > /tmp/medivoice_ollama_pull_fallback.log 2>&1",
            f"{FALLBACK_MODEL} downloaded"
        )
        if fallback_result.returncode != 0:
            raise RuntimeError("No demo model could be downloaded. Check Ollama logs in /tmp.")
        selected_model = FALLBACK_MODEL

print(f"[ok] Demo model selected: {selected_model}")

print("\n== Smoke test ==")
smoke = requests.post(
    f"{OLLAMA_HOST}/api/chat",
    json={
        "model": selected_model,
        "messages": [{"role": "user", "content": "Say exactly: Medi is ready"}],
        "stream": False,
        "options": {"num_predict": 20},
    },
    timeout=60,
)
smoke.raise_for_status()
print(f"[ok] Model working - Medi: {smoke.json().get('message', {}).get('content', '')}")

print("\n== Browser-safe proxy ==")
class OllamaProxyHandler(BaseHTTPRequestHandler):
    def _set_headers(self, status_code=200, content_type="application/json"):
        self.send_response(status_code)
        self.send_header("Access-Control-Allow-Origin", "*")
        self.send_header("Access-Control-Allow-Methods", "GET, POST, OPTIONS")
        self.send_header("Access-Control-Allow-Headers", "Content-Type")
        self.send_header("Content-Type", content_type)
        self.end_headers()

    def do_OPTIONS(self):
        self._set_headers(204, "text/plain")

    def do_GET(self):
        if self.path in ("/", "/health"):
            self._set_headers(200)
            payload = {
                "status": "ok",
                "service": "medivoice-proxy",
                "model": selected_model,
                "repo_mode": APP_MODE,
            }
            self.wfile.write(json.dumps(payload).encode("utf-8"))
            return
        if self.path == "/api/tags":
            try:
                upstream = requests.get(f"{OLLAMA_HOST}/api/tags", timeout=30)
                self._set_headers(upstream.status_code)
                self.wfile.write(upstream.content)
            except Exception as exc:
                self._set_headers(502)
                self.wfile.write(json.dumps({"error": str(exc)}).encode("utf-8"))
            return
        self._set_headers(404)
        self.wfile.write(json.dumps({"error": "Not found"}).encode("utf-8"))

    def do_POST(self):
        if self.path != "/api/chat":
            self._set_headers(404)
            self.wfile.write(json.dumps({"error": "Not found"}).encode("utf-8"))
            return
        try:
            content_length = int(self.headers.get("Content-Length", "0"))
            body = self.rfile.read(content_length)
            upstream = requests.post(
                f"{OLLAMA_HOST}/api/chat",
                data=body,
                headers={"Content-Type": self.headers.get("Content-Type", "application/json")},
                timeout=180,
            )
            self._set_headers(upstream.status_code)
            self.wfile.write(upstream.content)
        except Exception as exc:
            self._set_headers(502)
            self.wfile.write(json.dumps({"error": str(exc)}).encode("utf-8"))

    def log_message(self, format, *args):
        return


proxy_server = ThreadingHTTPServer(("0.0.0.0", PROXY_PORT), OllamaProxyHandler)
threading.Thread(target=proxy_server.serve_forever, daemon=True).start()
print(f"[ok] Browser-safe proxy running on port {PROXY_PORT}")

print("\n== Optional public tunnel ==")
OLLAMA_URL = None
NGROK_ERROR = None
ngrok_token = ""

try:
    from pyngrok import ngrok

    ngrok_token = os.environ.get("NGROK_AUTHTOKEN", "").strip()
    if not ngrok_token:
        try:
            from kaggle_secrets import UserSecretsClient
            ngrok_token = UserSecretsClient().get_secret("NGROK_AUTHTOKEN").strip()
            print("[ok] NGROK_AUTHTOKEN loaded from Kaggle Secrets")
        except Exception:
            ngrok_token = ""

    if ngrok_token:
        ngrok.set_auth_token(ngrok_token)
        public_url = ngrok.connect(PROXY_PORT, bind_tls=True)
        OLLAMA_URL = public_url.public_url.rstrip("/") + "/api/chat"
        print(f"[ok] Browser-safe Ollama Public URL: {OLLAMA_URL}")
        print("Paste this exact URL into the PWA Custom Ollama URL field.")
    else:
        print("[info] NGROK_AUTHTOKEN not found")
        print("[info] Add NGROK_AUTHTOKEN in Kaggle Secrets, restart the session, and rerun this notebook for phone/browser access.")
except Exception as exc:
    NGROK_ERROR = str(exc)
    print("[warn] ngrok public tunnel could not be created.")
    print(f"ngrok error: {NGROK_ERROR}")

print("\n== Live consultation demo ==")
from app.core.function_caller import run_agent
from app.utils.local_db import get_stats, init_db, log_visit

init_db()

demo_cases = [
    ("English triage", "Child has high fever for 3 days and not eating"),
    ("Roman Urdu triage", "Bachche ko 3 din se tez bukhar hai aur woh kuch nahi kha raha"),
    ("Hausa triage", "Yaro yana da zazzabi tsawon kwanaki 3 kuma baya cin abinci"),
    ("Dosage routing", "What is the paracetamol dosage for a 15kg child?"),
    ("Referral routing", "Where is the nearest hospital in Punjab?"),
    ("Emergency verdict", "Patient is unconscious and not breathing"),
]

print("=" * 64)
print("MEDIVOICE - LIVE CONSULTATION DEMO")
print(f"Offline-first AI | Primary {DEFAULT_MODEL} | Fallback {FALLBACK_MODEL} | UI languages {len(SUPPORTED_LANGUAGES)}")
print("=" * 64)

def derive_severity_and_action(response_text):
    upper = response_text.upper()
    if upper.startswith("EMERGENCY"):
        return "EMERGENCY", "FORAN HOSPITAL"
    if upper.startswith("REFER TO CLINIC") or upper.startswith("REFER"):
        return "REFER", "CLINIC REFER KAREIN"
    return "HOME CARE", "GHAR PE DEKHBHAL"


history = []
for label, message in demo_cases:
    print(f"\n[{label}]")
    print(f"CHW   : {message}")
    result = run_agent(message, history)
    history = result["history"]
    print(f"Medi  : {result['response']}")
    severity, action = derive_severity_and_action(result["response"])
    log_visit(
        symptoms=message,
        severity=severity,
        action=action,
        language=label.lower(),
        tool_used=result.get("tool_used"),
        response=result["response"],
    )
    if result.get("tool_used"):
        print(f"Tool  : {result['tool_used']}")
        print(f"Result: {result['tool_result']}")
    if result.get("explanation"):
        print(f"Why   : {json.dumps(result['explanation'], ensure_ascii=False)}")
    print("-" * 44)

stats = get_stats()
print(
    f"\nSession Stats - Total:{stats['total_visits']} | Emergency:{stats['emergencies']} | Refer:{stats['referrals']} | Home:{stats['home_care']}"
)

if OLLAMA_URL:
    pwa_mode = f"""
    <div style=\"background:#e8f7ef;border-radius:16px;padding:16px;font-size:0.84rem;color:#21413d;border:1px solid #b7ddc7;\">
      <b>Use with Kaggle + ngrok proxy:</b><br><br>
      1. Open <b>https://hamnamgl.github.io/Medivoice</b><br>
      2. Paste this exact URL into <b>Custom Ollama URL</b><br>
      3. <code>{OLLAMA_URL}</code><br>
      4. Tap <b>Save URL</b> and start chatting<br><br>
      This browser-safe proxy avoids the direct cross-origin Ollama issue.
    </div>
    """
else:
    pwa_mode = f"""
    <div style=\"background:#fff6df;border-radius:16px;padding:16px;font-size:0.84rem;color:#7a481f;border:1px solid #f0d28d;\">
      <b>ngrok setup required for phone/browser demo:</b><br><br>
      1. Create and verify an ngrok account<br>
      2. Add <code>NGROK_AUTHTOKEN</code> to Kaggle Secrets<br>
      3. Restart the notebook session and rerun this cell<br><br>
      <b>Current ngrok error:</b><br>
      <code>{NGROK_ERROR}</code>
    </div>
    """

display(HTML(f"""
<div style=\"font-family:Trebuchet MS, Segoe UI, sans-serif;background:linear-gradient(135deg,#21413d,#2b5160);color:#f5fbf7;padding:24px;border-radius:20px;border:1px solid rgba(255,255,255,0.12);margin:16px 0;box-shadow:0 18px 45px rgba(16,24,40,0.18);\">
  <div style=\"display:inline-block;padding:6px 10px;border-radius:999px;background:rgba(255,255,255,0.12);font-size:0.72rem;font-weight:700;letter-spacing:0.04em;text-transform:uppercase;\">Repo-aligned demo mode</div>
  <h2 style=\"margin:12px 0 8px 0;color:#ffffff;\">MediVoice PWA - Install on Android</h2>
  <p style=\"color:#d6efe4;font-size:0.9rem;line-height:1.55;margin-bottom:16px;\">
    Kaggle notebook is the reproducible demo path. True offline deployment runs the same MediVoice stack locally with Ollama on-device or on the same local network.
  </p>
  <div style=\"display:flex;gap:10px;flex-wrap:wrap;margin-bottom:18px;\">
    <div style=\"background:rgba(255,255,255,0.1);padding:10px 12px;border-radius:14px;\"><b>Primary:</b> {DEFAULT_MODEL}</div>
    <div style=\"background:rgba(255,255,255,0.1);padding:10px 12px;border-radius:14px;\"><b>Fallback:</b> {FALLBACK_MODEL}</div>
    <div style=\"background:rgba(255,255,255,0.1);padding:10px 12px;border-radius:14px;\"><b>Languages:</b> {len(SUPPORTED_LANGUAGES)}</div>
  </div>
  <a href=\"https://hamnamgl.github.io/Medivoice\" target=\"_blank\"
     style=\"display:inline-block;background:#dff4ea;color:#21413d;padding:12px 28px;border-radius:999px;font-weight:700;text-decoration:none;font-size:0.96rem;margin-bottom:18px;\">
    Open MediVoice PWA
  </a>
  {pwa_mode}
  <div style=\"margin-top:14px;font-size:0.8rem;color:#d6efe4;\">
    Repo: <a href=\"https://github.com/hamnamgl/Medivoice\" style=\"color:#ffffff;\">github.com/hamnamgl/Medivoice</a>
  </div>
</div>
"""))

print("\nMediVoice demo complete.")
print("Repo : https://github.com/hamnamgl/Medivoice")
print("PWA  : https://hamnamgl.github.io/Medivoice")
if OLLAMA_URL:
    print(f"Tunnel: {OLLAMA_URL}")
else:
    print("Tunnel: not available - set NGROK_AUTHTOKEN and rerun for public PWA access")
